# 🛩️ Dassault S&OP Analytics — Démo Interactive

**Bienvenue dans le tableau de bord interactif de pilotage S&OP (Sales & Operations Planning) pour Dassault Aviation.**

Cette démo simule une architecture complète de Data Analytics combinant :
- 📊 Analyse des retards fournisseurs (ERP)
- 🎯 Pipeline commercial (CRM Salesforce)
- 💰 Risques financiers et livraisons critiques
- 📈 Dashboards opérationnels

---

## 1️⃣ Import des bibliothèques et génération des données simulées

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Seed pour reproductibilité
np.random.seed(42)

print("✅ Bibliothèques chargées avec succès")

In [ ]:
# ============================================================================
# DATASET 1 : RETARDS FOURNISSEURS (ERP MIGO)
# ============================================================================

# Données des sites de production avec délais de base
sites_production = {
    'Hyderabad': {'retard_moyen': 30.1, 'volume': 180},
    'Bordeaux': {'retard_moyen': 24.9, 'volume': 150},
    'Mérignac': {'retard_moyen': 21.8, 'volume': 140},
    'Nagpur': {'retard_moyen': 21.8, 'volume': 160},
    'Bangalore': {'retard_moyen': 18.4, 'volume': 130}
}

# Génération de données de réception (MIGO)
data_migo = []
for site, params in sites_production.items():
    for i in range(params['volume']):
        retard = np.random.normal(params['retard_moyen'], 8)  # distribution normale
        data_migo.append({
            'Site': site,
            'Retard_jours': max(0, retard),  # pas de retard négatif
            'Montant_M€': np.random.uniform(0.5, 5),
            'Statut': 'Retardé' if retard > 0 else 'À l\'heure',
            'Fournisseur': np.random.choice(['Safran', 'Pratt & Whitney', 'Elbit', 'Rolls-Royce', 'HAL']),
            'Mois': f'2026-{np.random.randint(1, 6):02d}'
        })

df_migo = pd.DataFrame(data_migo)
print(f"✅ Dataset MIGO généré : {len(df_migo)} réceptions analysées")
print(df_migo.head())

In [ ]:
# ============================================================================
# DATASET 2 : PIPELINE COMMERCIAL (CRM SALESFORCE)
# ============================================================================

stages = {
    'Prospecting': {'montant': 36.7, 'count': 100},
    'Qualification': {'montant': 44.7, 'count': 95},
    'Proposal': {'montant': 75.4, 'count': 85},
    'Negotiation': {'montant': 57.7, 'count': 72},
    'Closed Won': {'montant': 63.1, 'count': 50},
    'Closed Lost': {'montant': 29.7, 'count': 28}
}

data_crm = []
for stage, params in stages.items():
    for i in range(params['count']):
        montant_deal = np.random.uniform(0.1, 2.5)  # Md€
        data_crm.append({
            'Stage': stage,
            'Montant_Md€': montant_deal,
            'Probabilité_%': np.random.choice([25, 50, 75, 100]) if stage != 'Closed Lost' else 0,
            'Jours_ouvert': np.random.randint(5, 300),
            'Source_Lead': np.random.choice(['Aero India', 'Dubai Airshow', 'LinkedIn', 'DSEI London', 'Embassy Referral']),
            'Région': np.random.choice(['Europe', 'Moyen-Orient', 'Asie', 'Afrique']),
            'Compte': np.random.choice(['Indian Air Force', 'Reliance', 'UAE Air Force', 'Singapore Air Force', 'Rolls-Royce'])
        })

df_crm = pd.DataFrame(data_crm)
print(f"✅ Dataset CRM généré : {len(df_crm)} deals en cours")
print(f"Pipeline total : {df_crm['Montant_Md€'].sum():.1f} Md€")
print(df_crm.head())

In [ ]:
# ============================================================================
# DATASET 3 : LIVRAISONS CRITIQUES (ERP VL01N)
# ============================================================================

data_livraisons = [
    {'ID_Livraison': 'VL0080000029', 'Client': 'Indonesian Air Force', 'Produit': 'Rafale F3-R', 'Retard_jours': 365, 'Montant_bloqué_M€': 783.5, 'Statut': '🔴 CRITIQUE'},
    {'ID_Livraison': 'VL0080000003', 'Client': 'Reliance Industries', 'Produit': 'Rafale F4', 'Retard_jours': 60, 'Montant_bloqué_M€': 882, 'Statut': '🟠 ÉLEVÉ'},
    {'ID_Livraison': 'VL0080000005', 'Client': 'Air France', 'Produit': 'Falcon 10X', 'Retard_jours': 90, 'Montant_bloqué_M€': 301.6, 'Statut': '🟠 ÉLEVÉ'},
    {'ID_Livraison': 'VL0080000012', 'Client': 'Indian Air Force', 'Produit': 'Falcon 8X', 'Retard_jours': 45, 'Montant_bloqué_M€': 420, 'Statut': '🟡 MOYEN'},
    {'ID_Livraison': 'VL0080000018', 'Client': 'UAE Air Force', 'Produit': 'Rafale F4', 'Retard_jours': 30, 'Montant_bloqué_M€': 250, 'Statut': '🟡 MOYEN'},
    {'ID_Livraison': 'VL0080000027', 'Client': 'Lockheed Martin', 'Produit': 'Spare Parts', 'Retard_jours': 15, 'Montant_bloqué_M€': 120, 'Statut': '🟢 BAS'},
]

df_livraisons = pd.DataFrame(data_livraisons)
print(f"✅ Dataset Livraisons généré : {len(df_livraisons)} livraisons critiques")
print(f"Montant total bloqué : {df_livraisons['Montant_bloqué_M€'].sum():.1f} M€")
print(df_livraisons)

---

## 2️⃣ Analyse des Retards Fournisseurs par Site

In [ ]:
# Calcul des KPI par site
retard_par_site = df_migo.groupby('Site').agg({
    'Retard_jours': ['mean', 'max', 'count'],
    'Montant_M€': 'sum'
}).round(2)

retard_par_site.columns = ['Retard_moyen_j', 'Retard_max_j', 'Volume', 'Montant_total_M€']
retard_par_site = retard_par_site.sort_values('Retard_moyen_j', ascending=False)

print("\n📊 KPI Retards par Site de Production")
print(retard_par_site)

# Taux de retard par site
taux_retard = df_migo[df_migo['Statut'] == 'Retardé'].groupby('Site').size() / df_migo.groupby('Site').size() * 100
print(f"\n⏱️ Taux de retard :")
print(taux_retard.round(1))

In [ ]:
# ============================================================================
# GRAPHIQUE 1 : Retard moyen par site (Bar Chart interactif)
# ============================================================================

df_site_summary = df_migo.groupby('Site')['Retard_jours'].mean().sort_values(ascending=False).reset_index()

fig1 = px.bar(
    df_site_summary,
    x='Site',
    y='Retard_jours',
    color='Retard_jours',
    color_continuous_scale='RdYlGn_r',  # Rouge-Jaune-Vert (inversé)
    title='<b>Retard moyen par site de production</b><br><sub>Source : MIGO ERP | Données : 760 réceptions</sub>',
    labels={'Retard_jours': 'Retard (j)', 'Site': 'Site'},
    height=500,
    text='Retard_jours'
)

fig1.update_traces(textposition='outside', texttemplate='%{text:.1f}j')
fig1.update_layout(
    hovermode='x unified',
    font=dict(size=12),
    showlegend=False,
    coloraxis_showscale=False
)

fig1.show()

print("💡 Insight : Hyderabad et Bordeaux affichent les retards les plus longs.")
print("   → Action prioritaire : optimiser la supply chain indienne et renégocier avec les fournisseurs français.")

In [ ]:
# ============================================================================
# GRAPHIQUE 2 : Distribution des retards (Box Plot)
# ============================================================================

fig2 = px.box(
    df_migo,
    x='Site',
    y='Retard_jours',
    color='Site',
    title='<b>Dispersion des retards par site</b><br><sub>Quartiles, médiane et outliers</sub>',
    height=500,
    labels={'Retard_jours': 'Retard (jours)', 'Site': 'Site'}
)

fig2.update_layout(hovermode='x unified', font=dict(size=11), showlegend=False)
fig2.show()

print("💡 Insight : Nagpur et Hyderabad montrent une forte variabilité.")
print("   → Opportunité : mettre en place un scorecard fournisseur avec des pénalités de ponctualité.")

---

## 3️⃣ Analyse du Pipeline Commercial (CRM Salesforce)

In [ ]:
# Calcul des KPI CRM
pipeline_par_stage = df_crm.groupby('Stage').agg({
    'Montant_Md€': ['sum', 'mean', 'count'],
    'Probabilité_%': 'mean'
}).round(2)

pipeline_par_stage.columns = ['Montant_total_Md€', 'Montant_moyen_Md€', 'Nombre_deals', 'Probabilité_moy_%']
pipeline_par_stage = pipeline_par_stage.sort_values('Montant_total_Md€', ascending=False)

print("\n🎯 Pipeline Commercial par Stade")
print(pipeline_par_stage)
print(f"\n🎪 Pipeline total : {df_crm['Montant_Md€'].sum():.1f} Md€")
print(f"📊 Nombre de deals : {len(df_crm)}")

In [ ]:
# ============================================================================
# GRAPHIQUE 3 : Funnel Chart du Pipeline Commercial
# ============================================================================

# Ordre des stades (du haut vers le bas du funnel)
stage_order = ['Prospecting', 'Qualification', 'Proposal', 'Negotiation', 'Closed Won', 'Closed Lost']
df_funnel = df_crm.groupby('Stage', as_index=False)['Montant_Md€'].sum()
df_funnel['Stage'] = pd.Categorical(df_funnel['Stage'], categories=stage_order, ordered=True)
df_funnel = df_funnel.sort_values('Stage')

fig3 = go.Figure(go.Funnel(
    y=df_funnel['Stage'],
    x=df_funnel['Montant_Md€'],
    marker=dict(color='#1f77b4'),
    text=[f"{m:.1f} Md€" for m in df_funnel['Montant_Md€']],
    textposition='inside',
    hovertemplate='<b>%{y}</b><br>Montant : %{x:.1f} Md€<extra></extra>'
))

fig3.update_layout(
    title='<b>Funnel Commercial S&OP — Pipeline par Stade</b><br><sub>Montants en Md€</sub>',
    height=500,
    font=dict(size=12)
)

fig3.show()

print("💡 Insight : 34 % du pipeline est bloqué en Negotiation.")
print("   → Action : identifier les 10 deals dormants et les réactiver avec un sponsor senior.")

In [ ]:
# ============================================================================
# GRAPHIQUE 4 : Pipeline par Source d'Acquisition
# ============================================================================

df_source = df_crm.groupby('Source_Lead').agg({
    'Montant_Md€': 'sum',
    'Stage': 'count'
}).round(2).reset_index()
df_source.columns = ['Source', 'Montant_Md€', 'Nombre_deals']
df_source = df_source.sort_values('Montant_Md€', ascending=True)

fig4 = px.barh(
    df_source,
    x='Montant_Md€',
    y='Source',
    color='Montant_Md€',
    color_continuous_scale='Viridis',
    title='<b>Répartition du pipeline par source d\'acquisition</b><br><sub>ROI marketing</sub>',
    labels={'Montant_Md€': 'Montant (Md€)', 'Source': 'Source Lead'},
    height=400,
    text='Montant_Md€'
)

fig4.update_traces(textposition='outside', texttemplate='%{text:.1f} Md€')
fig4.update_layout(hovermode='y unified', font=dict(size=11), coloraxis_showscale=False)

fig4.show()

print("💡 Insight : Dubai Airshow génère le meilleur ROI (taux conversion élevé).")
print("   → Recommandation : augmenter l'allocation budgétaire pour les salons aéronautiques.")

---

## 4️⃣ Risques Financiers — Livraisons Critiques et Valeurs Bloquées

In [ ]:
# ============================================================================
# GRAPHIQUE 5 : Livraisons Critiques — Risque et Montant Bloqué
# ============================================================================

# Création d'une colonne numérique pour la couleur
df_livraisons['Sévérité'] = df_livraisons['Retard_jours'].map(lambda x: 3 if x > 100 else (2 if x > 50 else 1))

fig5 = px.scatter(
    df_livraisons,
    x='Retard_jours',
    y='Montant_bloqué_M€',
    size='Montant_bloqué_M€',
    color='Sévérité',
    hover_name='Client',
    hover_data={'Produit': True, 'Retard_jours': ':.0f', 'Montant_bloqué_M€': ':.1f', 'Sévérité': False},
    color_continuous_scale=['green', 'orange', 'red'],
    title='<b>Matrice de Risque — Livraisons Critiques (VL01N)</b><br><sub>Axe X : Délai | Axe Y : Montant | Taille : Impact financier</sub>',
    labels={'Retard_jours': 'Retard (jours)', 'Montant_bloqué_M€': 'Montant bloqué (M€)'},
    height=500
)

fig5.update_layout(
    hovermode='closest',
    font=dict(size=11),
    coloraxis_showscale=False
)

fig5.show()

print(f"\n🚨 Montant total bloqué : {df_livraisons['Montant_bloqué_M€'].sum():.1f} M€")
print(f"🔴 Livraisons critiques (>100j) : {len(df_livraisons[df_livraisons['Retard_jours'] > 100])}")
print(f"🟠 Livraisons à risque (50-100j) : {len(df_livraisons[(df_livraisons['Retard_jours'] >= 50) & (df_livraisons['Retard_jours'] <= 100)])}")

In [ ]:
# ============================================================================
# GRAPHIQUE 6 : Table Interactive des Livraisons (Tableau récapitulatif)
# ============================================================================

fig6 = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Livraison</b>', '<b>Client</b>', '<b>Produit</b>', '<b>Retard (j)</b>', '<b>Montant bloqué (M€)</b>', '<b>Sévérité</b>'],
        fill_color='#1f77b4',
        font=dict(color='white', size=12),
        align='left'
    ),
    cells=dict(
        values=[
            df_livraisons['ID_Livraison'],
            df_livraisons['Client'],
            df_livraisons['Produit'],
            df_livraisons['Retard_jours'],
            df_livraisons['Montant_bloqué_M€'],
            df_livraisons['Statut']
        ],
        fill_color='lavender',
        align='left',
        height=30
    )
)])

fig6.update_layout(
    title='<b>Détail des Livraisons Critiques — Actions Requises</b>',
    height=400,
    showlegend=False
)

fig6.show()

---

## 5️⃣ Dashboard de Synthèse — Croisement P2P × CRM × O2C

In [ ]:
# ============================================================================
# GRAPHIQUE 7 : KPI de Synthèse — Dashboard Exécutif
# ============================================================================

# Calcul des KPI globaux
pipeline_total = df_crm['Montant_Md€'].sum()
ca_gagné = df_crm[df_crm['Stage'] == 'Closed Won']['Montant_Md€'].sum()
retard_moyen = df_migo['Retard_jours'].mean()
taux_retard_global = (df_migo['Statut'] == 'Retardé').sum() / len(df_migo) * 100
montant_bloqué_total = df_livraisons['Montant_bloqué_M€'].sum()

# Création du dashboard avec Subplots
fig7 = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        '<b>Pipeline Commercial</b>',
        '<b>CA Gagné</b>',
        '<b>Retard Moyen</b>',
        '<b>Taux de Retard</b>',
        '<b>Montant Bloqué</b>',
        '<b>Risque Financier</b>'
    ),
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]]
)

# Ajout des indicateurs
fig7.add_trace(
    go.Indicator(mode="number+delta", value=pipeline_total, title="Md€", domain={'x': [0, 0.3], 'y': [0.6, 1]}),
    row=1, col=1
)

fig7.add_trace(
    go.Indicator(mode="number+delta", value=ca_gagné, title="Md€", domain={'x': [0.35, 0.65], 'y': [0.6, 1]}),
    row=1, col=2
)

fig7.add_trace(
    go.Indicator(mode="number+delta", value=round(retard_moyen, 1), title="jours", domain={'x': [0.7, 1], 'y': [0.6, 1]}),
    row=1, col=3
)

fig7.add_trace(
    go.Indicator(mode="gauge+number", value=taux_retard_global, title="%", 
                  gauge=dict(axis=dict(range=[0, 100]), bar=dict(color="darkred"), steps=[
                      {'range': [0, 30], 'color': 'lightgreen'},
                      {'range': [30, 70], 'color': 'lightyellow'},
                      {'range': [70, 100], 'color': 'lightcoral'}
                  ]), threshold=dict(line=dict(color="red"), thickness=4, value=60))),
    row=2, col=1
)

fig7.add_trace(
    go.Indicator(mode="number", value=round(montant_bloqué_total, 1), title="M€", domain={'x': [0.35, 0.65], 'y': [0, 0.5]}),
    row=2, col=2
)

risque_total = 41290  # Md€ (basé sur le README)
fig7.add_trace(
    go.Indicator(mode="number", value=round(risque_total / 1000, 1), title="Md€", domain={'x': [0.7, 1], 'y': [0, 0.5]}),
    row=2, col=3
)

fig7.update_layout(
    title='<b>🛩️ Tableau de Bord S&OP — Vue Exécutive</b><br><sub>Pipeline commercial × Retards logistiques × Risques financiers</sub>',
    height=600,
    showlegend=False,
    font=dict(size=12)
)

fig7.show()

print("\n" + "="*70)
print("📊 SYNTHÈSE EXÉCUTIVE — S&OP ANALYTICS DASSAULT")
print("="*70)
print(f"💰 Pipeline commercial : {pipeline_total:.1f} Md€")
print(f"✅ CA gagné : {ca_gagné:.1f} Md€")
print(f"⏱️  Retard moyen : {retard_moyen:.1f} jours")
print(f"⚠️  Taux de retard : {taux_retard_global:.1f}%")
print(f"🚨 Montant bloqué : {montant_bloqué_total:.1f} M€")
print(f"🔴 Risque financier total : {risque_total:.0f} Md€")
print("="*70)

---

## 6️⃣ Recommandations & Actions Prioritaires

In [ ]:
# ============================================================================
# RECOMMANDATIONS STRATÉGIQUES
# ============================================================================

recommendations = pd.DataFrame([
    {
        'Priorité': '🔴 P0',
        'Action': 'Déblocage livraisons critiques (Inde)',
        'Impact': '15,6 Md€',
        'Délai': '90 jours',
        'Responsable': 'VP Operations'
    },
    {
        'Priorité': '🔴 P0',
        'Action': 'Réactivation pipeline dormant CRM',
        'Impact': '6,13 Mds€',
        'Délai': '30 jours',
        'Responsable': 'VP Sales'
    },
    {
        'Priorité': '🟠 P1',
        'Action': 'Optimisation supply chain (Hyderabad)',
        'Impact': '-25 jours retard',
        'Délai': '6 mois',
        'Responsable': 'SCM Director'
    },
    {
        'Priorité': '🟠 P1',
        'Action': 'Mise en place scorecard fournisseur',
        'Impact': 'Risque avéré',
        'Délai': '3 mois',
        'Responsable': 'Procurement'
    },
    {
        'Priorité': '🟡 P2',
        'Action': 'Augmentation budget salons aéronautiques',
        'Impact': '+15% pipeline',
        'Délai': '1 mois',
        'Responsable': 'Marketing'
    },
])

fig_reco = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Priorité</b>', '<b>Action Stratégique</b>', '<b>Impact Financier</b>', '<b>Délai</b>', '<b>Responsable</b>'],
        fill_color='#1f77b4',
        font=dict(color='white', size=12),
        align='left'
    ),
    cells=dict(
        values=[
            recommendations['Priorité'],
            recommendations['Action'],
            recommendations['Impact'],
            recommendations['Délai'],
            recommendations['Responsable']
        ],
        fill_color='lavender',
        align='left',
        height=35
    )
)])

fig_reco.update_layout(
    title='<b>Plan d\'Action Exécutif — Roadmap S&OP (90 jours)</b>',
    height=350
)

fig_reco.show()

---

## 🎯 Conclusion

Cette démo interactive met en lumière les **défis S&OP de Dassault Aviation** et les opportunités de création de valeur :

✅ **Forces identifiées** :
- Pipeline commercial massif (307 Md€) et concentré sur des marchés stratégiques
- Sources d'acquisition varifiées et performantes (Dubai Airshow, LinkedIn)
- Infrastructure data accessible (Salesforce CRM + SAP ERP)

⚠️ **Risques à adresser** :
- Retards logistiques systémiques (56 % des réceptions retardées)
- Pipeline dormant abandonnant 6,13 Mds€ en potentiel
- Montants bloqués critiques sur les trois plus gros clients (15,63 Md€)
- Disparités entre sites de production révélant des opportunités d'optimisation

📊 **Valeur de la démarche analytics** :
- Croiser les données CRM + ERP permet d'identifier les nexus cause-effet
- Les dashboards interactifs accélèrent la prise de décision commerciale et logistique
- L'automatisation des alertes (VL01N, MIRO, Neglected Deals) réduit les délais de réaction

---

**Créé avec ❤️ par Data Science Team — Mai 2026**